In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
def build_phq8_lookup(phq8_csv_path: Path, site_project_id: str | None = None) -> pd.DataFrame:
    """Turn raw PHQ-8 item-level questionnaire rows into one score per
    participant per completion date (sum of the 8 item values: value0..value7).
    Optionally filter to a single site via its project_id (e.g. 'RADAR-MDD-CIBER-s1').
    """
    raw = pd.read_csv(phq8_csv_path)
    if site_project_id is not None:
        raw = raw[raw["project_id"] == site_project_id]

    value_cols = [f"value{i}" for i in range(8)]
    raw = raw.copy()
    raw["phq8_score"] = raw[value_cols].sum(axis=1)
    raw["completed_date"] = pd.to_datetime(raw["time_completed_local"]).dt.normalize()

    return raw[["participant_name", "completed_date", "phq8_score"]].rename(
        columns={"participant_name": "participant_id"}
    )

In [ ]:
def attach_phq8_within_window(
    features_df: pd.DataFrame,
    phq8_lookup: pd.DataFrame,
    date_col: str = "Date",
    participant_col: str = "participant_id",
    window_days: int = 7) -> pd.DataFrame:
    """For each recording row, find the nearest PHQ-8 completion for the same
    participant and keep it only if within `window_days` of the recording date.
    Returns a copy of features_df with its PHQ8/phq8_score column filled in.
    """
    df = features_df.copy()
    df["_recording_date"] = pd.to_datetime(df[date_col])
    df["_orig_order"] = np.arange(len(df))

    left = df.sort_values("_recording_date")
    right = phq8_lookup.sort_values("completed_date")

    merged = pd.merge_asof(
        left,
        right,
        left_on="_recording_date",
        right_on="completed_date",
        by=participant_col,
        direction="nearest",
        tolerance=pd.Timedelta(days=window_days))

    merged = merged.sort_values("_orig_order").reset_index(drop=True)
    n_matched = merged["phq8_score"].notna().sum()
    print(f"Matched {n_matched} / {len(merged)} recordings to a PHQ-8 score within {window_days} days")

    merged = merged.drop(columns=["_recording_date", "_orig_order", "completed_date"])
    return merged

In [ ]:
PROJECT = Path("/Users/k1777551/JansCode/ASMHI-Research-Project")
RADAR_CSV_DIR = PROJECT / "data/processed"

phq8_lookup_site = build_phq8_lookup(
    RADAR_CSV_DIR / "phq8_data.csv",
    site_project_id="RADAR-MDD-VUmc-s1",
)

site_df = pd.read_csv(RADAR_CSV_DIR / "radar_model_dataset_raw_features-VUmc.csv")
site_df = attach_phq8_within_window(site_df, phq8_lookup_site, date_col="Date")

# Replace the old empty PHQ8 column with the matched score, then save
site_df["PHQ8"] = site_df["phq8_score"]
site_df = site_df.drop(columns=["phq8_score"])
site_df.to_csv(RADAR_CSV_DIR / "radar_model_dataset_raw_features-VUmc.csv", index=False)